# 00 — Base HSI canonique (tâches 03 à 05)


Ce notebook orchestre uniquement les fonctions du package. Toute configuration scientifique vient de `experiment_config.py`.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

from src import experiment_config as cfg
from src.data.database import (
    build_image_summary,
    build_minimal_nir_uco_object_database,
    build_object_summary,
    build_raw_image_manifest,
    preprocess_nir_uco_cube,
    validate_raw_image_manifest,
)
from src.io.database_h5 import (
    build_database_manifest,
    database_content_hash,
    load_nir_uco_h5,
    save_nir_uco_h5,
    validate_nir_uco_h5,
)
from src.io.dataload import load_mat_file
from src.protocol_governance import build_protocol_configuration, sha256_payload
from src.workflows.quality_check import build_segmentation_diagnostics_table

RAW_PATH = PROJECT_ROOT.joinpath(*cfg.RAW_MAT_RELATIVE_PATH)
RESULTS_DIR = PROJECT_ROOT.joinpath(*cfg.DATABASE_RESULTS_RELATIVE_DIR)
H5_PATH = PROJECT_ROOT.joinpath(*cfg.DATABASE_H5_RELATIVE_PATH)
OVERRIDE_DIR = PROJECT_ROOT.joinpath(*cfg.SEGMENTATION_OVERRIDE_RELATIVE_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = {
    key: RESULTS_DIR / filename
    for key, filename in cfg.DATABASE_OUTPUT_FILENAMES.items()
}


In [2]:
raw_data, load_info = load_mat_file(RAW_PATH, return_metadata=True)
raw_manifest, parsing_errors = build_raw_image_manifest(
    raw_data,
    expected_band_count=cfg.N_BANDS_RAW,
    strict_scientific_role=True,
)
raw_manifest.to_parquet(OUTPUT["raw_image_manifest"], index=False)
parsing_errors.to_parquet(OUTPUT["metadata_parsing_errors"], index=False)
validate_raw_image_manifest(
    raw_manifest,
    require_finite=True,
    require_known_role=True,
    require_common_band_count=True,
)
if not parsing_errors.empty:
    raise RuntimeError(
        "Construction canonique bloquée: des cubes HSI ne possèdent pas "
        "un rôle scientifique reconnu. Voir metadata_parsing_errors.parquet."
    )


In [3]:
raw_axis = np.linspace(
    cfg.SPECTRAL_START_NM,
    cfg.SPECTRAL_END_NM,
    cfg.N_BANDS_RAW,
)
wavelengths = raw_axis[cfg.N_REMOVE_START:cfg.N_STOP_END]
object_db, image_db = build_minimal_nir_uco_object_database(
    raw_data,
    selected_keys=cfg.DATABASE_SELECTED_KEYS,
    preprocess_func=preprocess_nir_uco_cube,
    n_remove_start=cfg.N_REMOVE_START,
    n_stop_end=cfg.N_STOP_END,
    wavelengths=wavelengths,
    data_mode=cfg.DATA_MODE,
    min_area=cfg.OBJECT_MIN_AREA,
    split=cfg.DATABASE_FORCED_SPLIT,
    skip_unknown=cfg.DATABASE_SKIP_UNKNOWN,
    segmentation_kwargs=cfg.SEGMENTATION_KWARGS,
    segmentation_overrides_dir=OVERRIDE_DIR,
)


Processing alm1pea1_sb | kind=mixture | components={'almond': {'batch': 1, 'token': 'alm'}, 'peanut': {'batch': 1, 'token': 'pea'}}


c:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\src\data\segmentation.py:120: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = morphology.remove_small_objects(mask, min_size=min_area)


  -> 42 objects detected
Processing alm1pea2_sb | kind=mixture | components={'almond': {'batch': 1, 'token': 'alm'}, 'peanut': {'batch': 2, 'token': 'pea'}}
  -> 40 objects detected
Processing alm1pea3_sb | kind=mixture | components={'almond': {'batch': 1, 'token': 'alm'}, 'peanut': {'batch': 3, 'token': 'pea'}}
  -> 40 objects detected
Processing alm1pea4_sb | kind=mixture | components={'almond': {'batch': 1, 'token': 'alm'}, 'peanut': {'batch': 4, 'token': 'pea'}}
  -> 27 objects detected
Processing alm2pea1_sb | kind=mixture | components={'almond': {'batch': 2, 'token': 'alm'}, 'peanut': {'batch': 1, 'token': 'pea'}}
  -> 40 objects detected
Processing alm2pea2_sb | kind=mixture | components={'almond': {'batch': 2, 'token': 'alm'}, 'peanut': {'batch': 2, 'token': 'pea'}}
  -> 40 objects detected
Processing alm2pea3_sb | kind=mixture | components={'almond': {'batch': 2, 'token': 'alm'}, 'peanut': {'batch': 3, 'token': 'pea'}}
  -> 44 objects detected
Processing alm2pea4_sb | kind=mix

In [4]:
configuration_hash = sha256_payload(build_protocol_configuration())
memory_hash = database_content_hash(object_db, image_db)
save_nir_uco_h5(
    object_db,
    image_db,
    H5_PATH,
    include_heavy_object_arrays=cfg.DATABASE_INCLUDE_HEAVY_OBJECT_ARRAYS,
    raw_file_sha256=load_info["file_sha256"],
    protocol_version=cfg.PROTOCOL_VERSION,
    configuration_hash=configuration_hash,
    content_hash=memory_hash,
    atomic=True,
)
validation = validate_nir_uco_h5(
    H5_PATH,
    expected_image_db=image_db,
    expected_object_db=object_db,
    deep=True,
    return_report=True,
)
if not validation["passed"].all():
    raise RuntimeError(validation.loc[~validation["passed"]].to_dict("records"))
reloaded_objects, reloaded_images = load_nir_uco_h5(
    H5_PATH,
    reconstruct_heavy_object_arrays=True,
)
if database_content_hash(reloaded_objects, reloaded_images) != memory_hash:
    raise RuntimeError("Le hash logique mémoire/HDF5 diffère après relecture.")


In [5]:
image_summary = build_image_summary(image_db)
object_summary = build_object_summary(object_db)
segmentation_diagnostics = build_segmentation_diagnostics_table(
    object_db,
    image_db,
)
database_manifest = build_database_manifest(
    object_db,
    image_db,
    H5_PATH,
    database_id="nir_uco_canonical",
    wavelength_mode=cfg.DEFAULT_WAVELENGTH_MODE,
    data_mode=cfg.DATA_MODE,
    protocol_version=cfg.PROTOCOL_VERSION,
    validation_report=validation,
)
image_summary.to_parquet(OUTPUT["image_summary"], index=False)
object_summary.to_parquet(OUTPUT["object_summary"], index=False)
segmentation_diagnostics.to_parquet(
    OUTPUT["segmentation_diagnostics"], index=False
)
database_manifest.to_parquet(OUTPUT["manifest"], index=False)
database_manifest


,database_id,wavelength_mode,data_mode,n_images,n_objects,n_bands,wavelength_min_nm,wavelength_max_nm,hdf5_valid,validation_failures,h5_schema_version,protocol_version,database_content_sha256,h5_file_sha256
0,nir_uco_canonical,non_noisy_all,reflectance,48,1262,63,960.735294,1702.0,True,0,1.0,8tracks_v1,f3b91bce30ef935346cb411743487c52fe309b74d1eb3d...,df382148759d1bc268046848ac4d7bcbe49a531c548eb9...
